# 08：采集文件与离线接收

默认先完成量化文件→模拟信道→采集文件→同步解调的完整闭环。结果明确标为 SIMULATED。拿到真实 ADC 文件后，在后面的可选单元格指定文件。

依赖：Python 3.8+、NumPy；完整解压本工具包后使用。

In [ ]:
from pathlib import Path
import sys, json
from datetime import datetime
from uuid import uuid4
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "afdm_lab").is_dir():
        ROOT = candidate.resolve()
        sys.path.insert(0, str(ROOT))
        break
else:
    raise RuntimeError("请先将整个 afdm_workbench.zip 解压；Notebook 不能脱离 afdm_lab 单独上传")
from afdm_lab.core import Config
from afdm_lab import workflows as work
cfg = Config()  # 默认 1 MHz 仅用于软件；不是自动测得的 RFDC 速率
out = ROOT / "afdm_runs" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_08_" + uuid4().hex[:6])
print("结果目录：", out)


In [ ]:
result = work.replay_file_loopback(out, cfg)
print(json.dumps(result, ensure_ascii=False, indent=2))
assert result["passed"], "检查失败：请保留 summary.json 和完整输出"


**判断依据**

默认应恢复三个 CRC 正确帧、PER=0。软件文件闭环通过不能证明射频通过。

结果保存在上面打印的目录与 `summary.json`。再次运行会建立新目录，不覆盖原结果。

**可选：真实采集文件的离线接收**

先按 docs/afdm-workbench.md 准备原始 I/Q 和相邻 JSON。不要使用 OFDM 的均衡后星座数据替代 ADC 采样。以下单元格默认跳过。

In [ ]:
REAL_CAPTURE = None  # 例如 Path("/home/xilinx/jupyter_notebooks/captures/rf_rx.iq16")
if REAL_CAPTURE is not None:
    actual_out = out.with_name(out.name + "_actual")
    actual = work.decode_file(REAL_CAPTURE, actual_out)
    print(json.dumps(actual, ensure_ascii=False, indent=2))
else:
    print("真实射频接收：未执行")
